# Thermal Stack

Le pipeline productif est centralise dans `thermal_stack.py`. Ce notebook execute la construction/export, puis conserve uniquement les affichages graphiques.


In [ ]:
# try:
#     result = build_thermal_stack(ThermalStackConfig())
# except PermissionError as exc:
#     print(f"Excel export failed; rebuilding without export. Details: {exc}")
#     result = build_thermal_stack(ThermalStackConfig(export_excel=False))

# print(f"Delivery month: {result.delivery_month:%Y-%m}")
# print(f"Thermal stack rows: {len(result.merit_order):,}")
# print(f"Excel export: {result.export_path}")

Delivery month: 2026-04
Thermal stack rows: 363
Excel export: C:\Develop\data\Japan\thermalStack.xlsx


In [10]:
"""Build the Japan thermal stack from MCP-catalogued datasets.

Auteur : Fourneret Leonard
Date de creation ou modification : 2026-05-12
"""

from __future__ import annotations

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

import data_loader

In [11]:
FAR_FUTURE = pd.Timestamp("2100-01-01")
EAST_JAPAN_AREAS = ("Hokkaido", "Tohoku", "Tokyo")
WEST_JAPAN_AREAS = ("Hokuriku", "Kansai", "Chugoku", "Shikoku", "Kyushu", "Chubu")
DEFAULT_PRODUCTION_TYPES = ("LNG", "Coal", "Hydro", "Oil", "Nuclear", "Biomass")
DEFAULT_STACK_FUELS = ("LNG", "Coal", "Oil", "Nuclear")

MMBTU_TO_MWH = 0.293071
DEFAULT_EFFICIENCY_BASIS = "LHV"
DEFAULT_FUEL_PRICE_BASIS = "HHV"
HHV_PER_LHV_BY_FUEL = {
    "LNG": 1.108,
    "Coal": 1.055,
    "Oil": 1.060,
}
VOM_USD_PER_MWH = {
    "LNG": 3.0,
    "Coal": 4.0,
    "Oil": 6.0,
    "Nuclear": 7.0,
}

In [12]:
OUTAGE_DETAIL_MAPPING = {
    "停止・定期検査等": {"severity": "full", "driver": "maintenance", "family": "Planned Maintenance"},
    "停止・長期計画停止": {"severity": "full", "driver": "maintenance", "family": "Planned Shutdown"},
    "停止・設備故障": {"severity": "full", "driver": "technical", "family": "Equipment Failure"},
    "停止・送電線等制約": {"severity": "full", "driver": "grid", "family": "Grid Constraint"},
    "停止・燃料制約": {"severity": "full", "driver": "fuel", "family": "Fuel Constraint"},
    "停止・その他": {"severity": "full", "driver": "other", "family": "Other"},
    "低下・設備故障": {"severity": "partial", "driver": "technical", "family": "Equipment Failure"},
    "低下・送電線等制約": {"severity": "partial", "driver": "grid", "family": "Grid Constraint"},
    "低下・燃料制約": {"severity": "partial", "driver": "fuel", "family": "Fuel Constraint"},
    "低下・その他": {"severity": "partial", "driver": "other", "family": "Other"},
}

MERIT_ORDER_COLORS = {
    "Nuclear": "#9467bd",
    "Coal": "#333333",
    "LNG": "#ff8c00",
    "Oil": "#8b4513",
}

In [13]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ThermalStackConfig:
    """Configuration for the thermal stack pipeline."""
    production_start: str | pd.Timestamp = "2024-03-01"
    outage_start: str | pd.Timestamp = "2024-03-15"
    forecast_start: str | pd.Timestamp | None = None
    delivery_month: str | pd.Timestamp | None = None
    export_excel: bool = True
    export_dataset_key: str = data_loader.THERMAL_STACK_EXPORT_DATASET_KEY
    load_unit_production: bool = True
    nuclear_efficiency_pct: float = 35.1
    outage_min_duration_days: float = 0.5
    excluded_areas: tuple[str, ...] = ("Okinawa",)
    production_types: tuple[str, ...] = DEFAULT_PRODUCTION_TYPES
    stack_fuels: tuple[str, ...] = DEFAULT_STACK_FUELS


@dataclass(frozen=True)
class ThermalStackResult:
    """Container returned by build_thermal_stack()."""
    merit_order: pd.DataFrame
    marginal_costs: pd.DataFrame
    active_units: pd.DataFrame
    adjusted_units: pd.DataFrame
    thermal_registry: pd.DataFrame
    nuclear_registry: pd.DataFrame
    outages_thermal: pd.DataFrame
    outages_nuclear: pd.DataFrame
    outages_all: pd.DataFrame
    fuel_cocktail: pd.DataFrame
    unit_production: pd.DataFrame
    delivery_month: pd.Timestamp
    capacity_summary: dict
    export_path: Path | None

In [14]:
def _require_columns(frame: pd.DataFrame, required_columns: Iterable[str], frame_name: str) -> None:
    """Raise error when required columns missing."""
    missing = [column for column in required_columns if column not in frame.columns]
    if missing:
        raise ValueError(f"{frame_name} is missing required columns: {missing}")


def _timestamp(value: str | pd.Timestamp | None, default: pd.Timestamp | None = None) -> pd.Timestamp:
    """Convert value to pd.Timestamp."""
    if value is None:
        if default is None:
            raise ValueError("A timestamp value is required when no default is provided.")
        return pd.Timestamp(default)
    return pd.Timestamp(value)


def _month_start(value: str | pd.Timestamp | None) -> pd.Timestamp:
    """Return first day of requested month."""
    timestamp = pd.Timestamp("today").normalize() if value is None else pd.Timestamp(value)
    return timestamp.to_period("M").to_timestamp()


def classify_region(area: object) -> str | float:
    """Classify Japan market area into East or West."""
    if area in EAST_JAPAN_AREAS:
        return "East"
    if area in WEST_JAPAN_AREAS:
        return "West"
    return np.nan


def _add_east_west(frame: pd.DataFrame, area_column: str) -> pd.DataFrame:
    """Add EastWest column from market-area column."""
    result = frame.copy()
    if area_column in result.columns:
        result["EastWest"] = result[area_column].apply(classify_region)
    return result

In [15]:
def load_filtered_unit_production(config: ThermalStackConfig) -> pd.DataFrame:
    """Load and filter HJKS unit production through data_loader."""
    if not config.load_unit_production:
        return pd.DataFrame()

    unit_production = data_loader.load_unit_production()
    _require_columns(unit_production, ["Date", "Type", "Market Area"], "unit_production")
    unit_production = unit_production.copy()
    unit_production["Date"] = pd.to_datetime(unit_production["Date"], errors="coerce")
    unit_production = unit_production[unit_production["Type"].isin(config.production_types)]
    unit_production = unit_production[
        unit_production["Date"] >= _timestamp(config.production_start)
    ].copy()
    return _add_east_west(unit_production, "Market Area").reset_index(drop=True)


def load_thermal_registry(config: ThermalStackConfig) -> pd.DataFrame:
    """Load and filter thermal efficiency registry."""
    thermal = data_loader.load_thermal_efficiency_registry().copy()
    _require_columns(
        thermal,
        [
            "plant_name_jp",
            "unit_digit",
            "fuel_class",
            "area_en",
            "capacity_mw",
            "plant_efficiency",
            "start_year",
            "retired_year",
        ],
        "thermal_registry",
    )
    thermal["start_year"] = pd.to_datetime(thermal["start_year"], errors="coerce")
    thermal["retired_year"] = pd.to_datetime(thermal["retired_year"], errors="coerce").fillna(FAR_FUTURE)
    thermal = thermal[thermal["retired_year"] >= _timestamp(config.outage_start)]
    if config.excluded_areas:
        thermal = thermal[~thermal["area_en"].isin(config.excluded_areas)]
    thermal["Type"] = thermal["fuel_class"]
    return _add_east_west(thermal.reset_index(drop=True), "area_en")


def prepare_nuclear_registry(
    nuclear_registry: pd.DataFrame,
    thermal_columns: Iterable[str],
    config: ThermalStackConfig,
) -> pd.DataFrame:
    """Convert nuclear registry rows to thermal-stack schema."""
    nuclear = nuclear_registry.copy()
    _require_columns(
        nuclear,
        [
            "unitAll_code",
            "unit30min_code",
            "plant_name_jp",
            "unit_digit",
            "area_en",
            "capacity_gw",
            "start_date",
            "end_date",
        ],
        "nuclear_registry",
    )
    nuclear["capacity_mw"] = pd.to_numeric(nuclear["capacity_gw"], errors="coerce") * 1000
    nuclear["fuel_class"] = "Nuclear"
    nuclear["Type"] = "Nuclear"
    nuclear["plant_efficiency"] = config.nuclear_efficiency_pct
    nuclear["start_year"] = pd.to_datetime(nuclear["start_date"], errors="coerce")
    nuclear["retired_year"] = pd.to_datetime(nuclear["end_date"], errors="coerce").fillna(FAR_FUTURE)

    columns = [column for column in thermal_columns if column in nuclear.columns]
    for column in [
        "unitAll_code",
        "unit30min_code",
        "plant_name_jp",
        "unit_digit",
        "area_en",
        "capacity_mw",
        "fuel_class",
        "Type",
        "plant_efficiency",
        "start_year",
        "retired_year",
        "owner",
    ]:
        if column in nuclear.columns and column not in columns:
            columns.append(column)
    nuclear = nuclear[columns].copy()
    return _add_east_west(nuclear.reset_index(drop=True), "area_en")

In [16]:
def filter_units_for_date(df_units: pd.DataFrame, target_date: str | pd.Timestamp) -> pd.DataFrame:
    """Filter units active on one date."""
    _require_columns(df_units, ["start_year", "retired_year"], "df_units")
    target = pd.Timestamp(target_date)
    mask = (df_units["start_year"] <= target) & (df_units["retired_year"] >= target)
    return df_units.loc[mask].copy()


def filter_units_for_forecast(
    df_units: pd.DataFrame,
    forecast_start: str | pd.Timestamp | None = None,
) -> pd.DataFrame:
    """Filter units not retired at forecast start."""
    _require_columns(df_units, ["retired_year"], "df_units")
    start = pd.Timestamp("today").normalize() if forecast_start is None else pd.Timestamp(forecast_start)
    return df_units.loc[df_units["retired_year"] >= start].copy()

In [17]:
def consolidate_outages(df_outages: pd.DataFrame) -> pd.DataFrame:
    """Merge HJKS update duplicates while preserving concurrent events."""
    if df_outages.empty:
        return df_outages.copy()

    _require_columns(
        df_outages,
        ["Name", "Unit Name", "OutageDetail", "From", "To", "Publish Time"],
        "df_outages",
    )
    outages = df_outages.sort_values(
        ["Name", "Unit Name", "OutageDetail", "From", "To"],
        ascending=[True, True, True, True, False],
    ).copy()

    keep_index = []
    for _, group in outages.groupby(["Name", "Unit Name", "OutageDetail"], sort=False):
        cluster_end = pd.Timestamp.min
        cluster_rows: list[int] = []
        for index, row in group.iterrows():
            if row["From"] > cluster_end:
                if cluster_rows:
                    keep_index.append(_latest_publish_index(outages, cluster_rows))
                cluster_rows = [index]
                cluster_end = row["To"]
            else:
                cluster_rows.append(index)
                cluster_end = max(cluster_end, row["To"])
        if cluster_rows:
            keep_index.append(_latest_publish_index(outages, cluster_rows))

    return outages.loc[keep_index].reset_index(drop=True)


def _latest_publish_index(df_outages: pd.DataFrame, indexes: list[int]) -> int:
    """Return index with latest publish timestamp."""
    return max(
        indexes,
        key=lambda index: (
            df_outages.loc[index, "Publish Time"]
            if pd.notna(df_outages.loc[index, "Publish Time"])
            else pd.Timestamp.min
        ),
    )


def annotate_outage_metadata(df_outages: pd.DataFrame) -> pd.DataFrame:
    """Add severity, driver, family and impact columns to outages."""
    outages = df_outages.copy()
    if outages.empty:
        return outages

    mapped = outages["OutageDetail"].map(lambda value: OUTAGE_DETAIL_MAPPING.get(value, {}))
    outages["severity"] = mapped.map(lambda value: value.get("severity", "unknown"))
    outages["driver"] = mapped.map(lambda value: value.get("driver", "unknown"))
    outages["family"] = mapped.map(lambda value: value.get("family", "Unknown"))
    if "Impact (GW)" in outages.columns:
        outages["impact_mw"] = pd.to_numeric(outages["Impact (GW)"], errors="coerce") * 1000
    return outages

In [18]:
def load_stack_outages(config: ThermalStackConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load and prepare thermal and nuclear outages."""
    outages = data_loader.load_outages().copy()
    _require_columns(outages, ["Type", "From", "To", "OutageDetail", "Name", "Unit Name", "Publish Time"], "outages")
    outages["From"] = pd.to_datetime(outages["From"], errors="coerce")
    outages["To"] = pd.to_datetime(outages["To"], errors="coerce")
    outages["Publish Time"] = pd.to_datetime(outages.get("Publish Time"), errors="coerce")
    outages = outages[outages["To"] >= _timestamp(config.outage_start)].copy()
    outages["duration_days"] = (outages["To"] - outages["From"]).dt.total_seconds() / 86400
    outages = outages[outages["duration_days"] >= config.outage_min_duration_days].copy()

    thermal = outages[~outages["Type"].isin(["Nuclear", "Hydro"])].copy()
    nuclear = outages[outages["Type"] == "Nuclear"].copy()
    nuclear = nuclear[~nuclear["Name"].astype(str).str.contains("柏崎刈羽", na=False)].copy()

    thermal = annotate_outage_metadata(consolidate_outages(thermal))
    nuclear = annotate_outage_metadata(consolidate_outages(nuclear))
    outages_all = pd.concat([thermal, nuclear], ignore_index=True)
    return thermal, nuclear, outages_all

In [19]:
def _compute_loss_mw(outage: pd.Series, capacity_mw: float) -> float:
    """Compute MW loss for one outage event."""
    severity = str(outage.get("severity", "partial")).lower()
    if severity == "full":
        return capacity_mw

    impact_mw = outage.get("impact_mw")
    if pd.notna(impact_mw) and impact_mw > 0:
        return min(float(impact_mw), capacity_mw)

    driver = str(outage.get("driver", "")).lower()
    if driver == "grid":
        return 0.3 * capacity_mw
    if driver == "fuel":
        return 0.5 * capacity_mw
    if driver == "technical":
        return 0.6 * capacity_mw
    return 0.4 * capacity_mw


def _match_unit_outages(unit: pd.Series, active_outages: pd.DataFrame) -> pd.DataFrame:
    """Match active outages to unit using available registry fields."""
    if active_outages.empty:
        return active_outages

    matched = active_outages.iloc[0:0]
    unit_code = unit.get("unitAll_code", np.nan)
    if pd.notna(unit_code) and "Plant Code" in active_outages.columns:
        matched = active_outages[active_outages["Plant Code"].astype(str) == str(unit_code)]

    if matched.empty and "Name" in active_outages.columns and "plant_name_jp" in unit.index:
        matched = active_outages[active_outages["Name"].astype(str) == str(unit["plant_name_jp"])]

    if matched.empty:
        return matched

    if "Unit Name" in matched.columns and pd.notna(unit.get("unit_digit", np.nan)):
        unit_digit = str(unit["unit_digit"])
        exact_unit = matched[matched["Unit Name"].astype(str) == unit_digit]
        return exact_unit if not exact_unit.empty else matched
    return matched

In [20]:
def build_adjusted_capacity(
    df_units: pd.DataFrame,
    df_outages_all: pd.DataFrame,
    target_date: str | pd.Timestamp,
) -> tuple[pd.DataFrame, dict]:
    """Adjust unit capacity by active outages for target date."""
    _require_columns(df_units, ["capacity_mw", "start_year", "retired_year"], "df_units")
    target = pd.Timestamp(target_date)
    adjusted = filter_units_for_date(df_units, target).copy()
    adjusted["Capacity_adj_GW"] = pd.to_numeric(adjusted["capacity_mw"], errors="coerce") / 1000

    if df_outages_all.empty:
        active_outages = df_outages_all.copy()
    else:
        active_outages = df_outages_all[
            (df_outages_all["From"] <= target) & (df_outages_all["To"] >= target)
        ].copy()

    full_stops = 0
    derates = 0
    for unit_index, unit in adjusted.iterrows():
        capacity_mw = pd.to_numeric(unit["capacity_mw"], errors="coerce")
        if pd.isna(capacity_mw) or capacity_mw <= 0:
            continue
        capacity_mw = float(capacity_mw)
        unit_outages = _match_unit_outages(unit, active_outages)
        if unit_outages.empty:
            continue

        total_loss_mw = 0.0
        for _, outage in unit_outages.iterrows():
            total_loss_mw += _compute_loss_mw(outage, capacity_mw)
            if str(outage.get("severity", "partial")).lower() == "full":
                full_stops += 1
            else:
                derates += 1

        adjusted.loc[unit_index, "Capacity_adj_GW"] -= min(total_loss_mw, capacity_mw) / 1000

    adjusted["Capacity_adj_GW"] = adjusted["Capacity_adj_GW"].clip(lower=0)
    gw_before = (pd.to_numeric(adjusted["capacity_mw"], errors="coerce") / 1000).sum()
    gw_after = adjusted["Capacity_adj_GW"].sum()
    adjusted = adjusted[adjusted["Capacity_adj_GW"] > 0].copy()

    summary = {
        "target_date": target,
        "n_full_stops": full_stops,
        "n_derates": derates,
        "gw_removed": gw_before - gw_after,
        "gw_before": gw_before,
        "gw_after": gw_after,
        "n_units_before": len(filter_units_for_date(df_units, target)),
        "n_units_after": len(adjusted),
    }
    return adjusted, summary


def build_adjusted_capacity_month(
    df_units: pd.DataFrame,
    df_outages_all: pd.DataFrame,
    target_month: str | pd.Timestamp,
) -> tuple[pd.DataFrame, dict]:
    """Adjust unit capacity for month using mid-month date."""
    mid_month = _month_start(target_month) + pd.Timedelta(days=14)
    return build_adjusted_capacity(df_units, df_outages_all, mid_month)

In [21]:
def _normalize_energy_basis(basis: object, default_basis: str) -> str:
    """Return canonical heating-value basis label."""
    if pd.isna(basis) or basis is None:
        return default_basis

    normalized = str(basis).strip().upper()
    alias_map = {
        "HHV": "HHV",
        "GCV": "HHV",
        "GROSS": "HHV",
        "HIGHER HEATING VALUE": "HHV",
        "LHV": "LHV",
        "NCV": "LHV",
        "NET": "LHV",
        "LOWER HEATING VALUE": "LHV",
    }
    if normalized not in alias_map:
        raise ValueError(f"Unsupported heating-value basis: {basis!r}")
    return alias_map[normalized]


def convert_efficiency_to_energy_basis(
    gen_type: str,
    efficiency_pct: float,
    from_basis: object = DEFAULT_EFFICIENCY_BASIS,
    to_basis: object = DEFAULT_FUEL_PRICE_BASIS,
) -> float:
    """Convert plant efficiency between LHV and HHV conventions."""
    if pd.isna(efficiency_pct) or efficiency_pct <= 0:
        return np.nan

    source_basis = _normalize_energy_basis(from_basis, DEFAULT_EFFICIENCY_BASIS)
    target_basis = _normalize_energy_basis(to_basis, DEFAULT_FUEL_PRICE_BASIS)
    if source_basis == target_basis or gen_type == "Nuclear":
        return float(efficiency_pct)

    hhv_per_lhv = HHV_PER_LHV_BY_FUEL.get(gen_type)
    if hhv_per_lhv is None or hhv_per_lhv <= 0:
        return np.nan

    efficiency = float(efficiency_pct)
    if source_basis == "LHV" and target_basis == "HHV":
        return efficiency / hhv_per_lhv
    if source_basis == "HHV" and target_basis == "LHV":
        return efficiency * hhv_per_lhv
    raise ValueError(f"Unsupported energy-basis conversion: {source_basis!r} -> {target_basis!r}")

In [22]:
def compute_mc_eur(
    gen_type: str,
    efficiency_pct: float,
    fuel_row: pd.Series | dict,
    efficiency_basis: object = DEFAULT_EFFICIENCY_BASIS,
    fuel_basis: object = DEFAULT_FUEL_PRICE_BASIS,
) -> float:
    """Compute marginal cost in EUR/MWh for one fuel and efficiency."""
    if pd.isna(efficiency_pct) or efficiency_pct <= 0:
        return np.nan

    eurusd = fuel_row.get("EURUSD", fuel_row.get("eurusd", np.nan))
    if pd.isna(eurusd) or eurusd <= 0:
        return np.nan

    if gen_type == "Nuclear":
        return VOM_USD_PER_MWH["Nuclear"] / eurusd

    if gen_type == "LNG":
        fuel = fuel_row.get("jlc_eurmwh", np.nan)
    elif gen_type == "Coal":
        fuel = fuel_row.get("coal_cif_eurmwh", np.nan)
    elif gen_type == "Oil":
        fuel = fuel_row.get("jcc_eurmwh", np.nan)
    else:
        return np.nan

    if pd.isna(fuel):
        return np.nan

    aligned_efficiency_pct = convert_efficiency_to_energy_basis(
        gen_type,
        efficiency_pct,
        from_basis=efficiency_basis,
        to_basis=fuel_basis,
    )
    if pd.isna(aligned_efficiency_pct) or aligned_efficiency_pct <= 0:
        return np.nan

    efficiency = aligned_efficiency_pct / 100.0
    variable_om = VOM_USD_PER_MWH.get(gen_type, 0.0) / eurusd
    return fuel / efficiency + variable_om


def build_marginal_costs(df_active_units: pd.DataFrame, df_cocktail: pd.DataFrame) -> pd.DataFrame:
    """Build monthly marginal costs for every active stack unit."""
    _require_columns(
        df_active_units,
        ["plant_name_jp", "unit_digit", "fuel_class", "plant_efficiency"],
        "df_active_units",
    )
    if df_cocktail.empty:
        raise ValueError("fuel_cocktail is empty; marginal costs cannot be computed.")

    monthly_fuels = df_cocktail.sort_index().groupby(df_cocktail.index.to_period("M")).last().copy()
    monthly_fuels["delivery"] = monthly_fuels.index.to_timestamp(how="start")

    records = []
    for _, fuel_row in monthly_fuels.iterrows():
        delivery = pd.Timestamp(fuel_row["delivery"])
        for _, plant in df_active_units.iterrows():
            mc_eur_mwh = compute_mc_eur(
                plant["fuel_class"],
                plant["plant_efficiency"],
                fuel_row,
                efficiency_basis=plant.get("plant_efficiency_basis", plant.get("efficiency_basis", DEFAULT_EFFICIENCY_BASIS)),
                fuel_basis=plant.get("fuel_price_basis", DEFAULT_FUEL_PRICE_BASIS),
            )
            aligned_efficiency_pct = convert_efficiency_to_energy_basis(
                plant["fuel_class"],
                plant["plant_efficiency"],
                from_basis=plant.get("plant_efficiency_basis", plant.get("efficiency_basis", DEFAULT_EFFICIENCY_BASIS)),
                to_basis=plant.get("fuel_price_basis", DEFAULT_FUEL_PRICE_BASIS),
            )
            records.append(
                {
                    "delivery": delivery,
                    "plant_name_jp": plant["plant_name_jp"],
                    "unit_digit": plant["unit_digit"],
                    "fuel_class": plant["fuel_class"],
                    "mc_eur_mwh": mc_eur_mwh,
                    "efficiency_pct": plant["plant_efficiency"],
                    "efficiency_pct_mc_basis": aligned_efficiency_pct,
                    "efficiency_basis": plant.get("plant_efficiency_basis", plant.get("efficiency_basis", DEFAULT_EFFICIENCY_BASIS)),
                    "fuel_price_basis": plant.get("fuel_price_basis", DEFAULT_FUEL_PRICE_BASIS),
                }
            )

    return pd.DataFrame.from_records(records)

In [23]:
def build_merit_order(
    delivery_month: str | pd.Timestamp,
    df_mc: pd.DataFrame,
    df_units: pd.DataFrame,
    region: str | None = None,
) -> pd.DataFrame:
    """Build sorted merit order for one delivery month."""
    _require_columns(df_units, ["plant_name_jp", "unit_digit", "fuel_class", "capacity_mw"], "df_units")
    _require_columns(df_mc, ["delivery", "plant_name_jp", "unit_digit", "fuel_class", "mc_eur_mwh"], "df_mc")

    delivery = _month_start(delivery_month)
    mc_month = df_mc[df_mc["delivery"] == delivery].copy()
    mc_columns = ["plant_name_jp", "unit_digit", "fuel_class", "mc_eur_mwh", "efficiency_pct"]
    mc_columns = [column for column in mc_columns if column in mc_month.columns]

    merged = df_units.merge(
        mc_month[mc_columns],
        on=["plant_name_jp", "unit_digit", "fuel_class"],
        how="left",
    )
    if region is not None:
        if "EastWest" not in merged.columns:
            merged = _add_east_west(merged, "area_en")
        merged = merged[merged["EastWest"] == region]

    merged["capacity_mw"] = pd.to_numeric(merged["capacity_mw"], errors="coerce")
    merged = merged.dropna(subset=["mc_eur_mwh"]).copy()
    merged = merged.dropna(subset=["capacity_mw"]).copy()
    merged = merged.sort_values("mc_eur_mwh").reset_index(drop=True)
    merged["cum_GW"] = merged["capacity_mw"].cumsum() / 1000
    return merged


def _select_merit_order_month(
    requested_month: pd.Timestamp,
    df_mc: pd.DataFrame,
    df_units: pd.DataFrame,
) -> tuple[pd.Timestamp, pd.DataFrame]:
    """Return non-empty merit order for requested or nearest month."""
    merit_order = build_merit_order(requested_month, df_mc, df_units, region=None)
    if not merit_order.empty:
        return requested_month, merit_order

    valid_months = sorted(df_mc.loc[df_mc["mc_eur_mwh"].notna(), "delivery"].dropna().unique())
    if not valid_months:
        raise ValueError("No available month with valid marginal-cost data.")

    fallback = max([month for month in valid_months if month <= requested_month], default=valid_months[0])
    fallback = pd.Timestamp(fallback)
    return fallback, build_merit_order(fallback, df_mc, df_units, region=None)

In [24]:
def export_thermal_stack(merit_order: pd.DataFrame, dataset_key: str) -> Path:
    """Export thermal stack to MCP-catalogued Excel path."""
    output_path = data_loader.resolve_dataset_output_path(dataset_key)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        merit_order.to_excel(writer, index=True)
    return output_path

In [25]:
def build_thermal_stack(config: ThermalStackConfig | None = None) -> ThermalStackResult:
    """Build and optionally export Japan thermal stack."""
    config = config or ThermalStackConfig()
    unit_production = load_filtered_unit_production(config)
    thermal_registry = load_thermal_registry(config)
    nuclear_source = data_loader.load_nuclear_registry()
    nuclear_registry = prepare_nuclear_registry(nuclear_source, thermal_registry.columns, config)

    stack_units = thermal_registry[thermal_registry["fuel_class"] != "Nuclear"].copy()
    stack_units = pd.concat([stack_units, nuclear_registry], ignore_index=True)

    outages_thermal, outages_nuclear, outages_all = load_stack_outages(config)
    forecast_start = pd.Timestamp("today").normalize() if config.forecast_start is None else pd.Timestamp(config.forecast_start)
    active_units = filter_units_for_forecast(stack_units, forecast_start=forecast_start)
    active_units = active_units[active_units["fuel_class"].isin(config.stack_fuels)].reset_index(drop=True)

    fuel_cocktail = data_loader.load_japan_fuel_cocktail()
    marginal_costs = build_marginal_costs(active_units, fuel_cocktail)

    requested_month = _month_start(config.delivery_month)
    delivery_month, merit_order = _select_merit_order_month(requested_month, marginal_costs, active_units)
    adjusted_units, capacity_summary = build_adjusted_capacity_month(stack_units, outages_all, delivery_month)

    export_path = export_thermal_stack(merit_order, config.export_dataset_key) if config.export_excel else None
    return ThermalStackResult(
        merit_order=merit_order,
        marginal_costs=marginal_costs,
        active_units=active_units,
        adjusted_units=adjusted_units,
        thermal_registry=thermal_registry,
        nuclear_registry=nuclear_registry,
        outages_thermal=outages_thermal,
        outages_nuclear=outages_nuclear,
        outages_all=outages_all,
        fuel_cocktail=fuel_cocktail,
        unit_production=unit_production,
        delivery_month=delivery_month,
        capacity_summary=capacity_summary,
        export_path=export_path,
    )

In [44]:
def plot_merit_order_plotly(
    merit_order: pd.DataFrame,
    region: str | list[str] | None = None,
    period: str | pd.Timestamp | None = None,
    title: str | None = None,
    height: int = 650,
):
    """Create interactive merit order chart, optionally filtered by region and period.
    
    Parameters
    ----------
    merit_order : pd.DataFrame
        Merit order dataframe from build_thermal_stack().
    region : str or list[str], optional
        Region(s) to plot. Can be any area from EAST_JAPAN_AREAS or WEST_JAPAN_AREAS.
        Examples: "Tokyo", "Kansai", ["Tokyo", "Hokkaido"]. If None, plots all-Japan.
    period : str or pd.Timestamp, optional
        Period label for display. Defaults to current month.
    title : str, optional
        Custom chart title. Auto-generated if not provided.
    height : int, default 650
        Chart height in pixels.
    """
    import plotly.graph_objects as go

    mo = merit_order.copy()
    _require_columns(mo, ["cum_GW", "capacity_mw", "mc_eur_mwh", "fuel_class"], "merit_order")
    
    # Filter by region if specified
    if region is not None:
        _require_columns(mo, ["area_en"], "merit_order")
        
        regions = region if isinstance(region, list) else [region]
        mo = mo[mo["area_en"].isin(regions)].copy()
        
        # Recalculate cumulative GW for filtered data
        mo = mo.sort_values("mc_eur_mwh").reset_index(drop=True)
        mo["cum_GW"] = mo["capacity_mw"].cumsum() / 1000
    
    # Set default period to current month
    if period is None:
        period = _month_start(None)
    else:
        period = pd.Timestamp(period)
    
    # Build title if not provided
    if title is None:
        region_label = " + ".join(region if isinstance(region, list) else [region]) if region else "All Japan"
        period_label = period.strftime("%B %Y")
        title = f"Merit Order ({region_label}) - {period_label}"
    
    fig = go.Figure()
    for fuel in ["Nuclear", "Coal", "LNG", "Oil"]:
        fuel_rows = mo[mo["fuel_class"] == fuel]
        for _, row in fuel_rows.iterrows():
            x0 = row["cum_GW"] - row["capacity_mw"] / 1000
            x1 = row["cum_GW"]
            y = row["mc_eur_mwh"]
            fig.add_trace(
                go.Scatter(
                    x=[x0, x1, x1, x0, x0],
                    y=[0, 0, y, y, 0],
                    fill="toself",
                    fillcolor=MERIT_ORDER_COLORS.get(fuel, "gray"),
                    opacity=0.7,
                    line=dict(color=MERIT_ORDER_COLORS.get(fuel, "gray"), width=0.5),
                    name=fuel,
                    legendgroup=fuel,
                    showlegend=False,
                    hoverinfo="skip",
                )
            )

    for fuel in ["Nuclear", "Coal", "LNG", "Oil"]:
        if fuel in set(mo["fuel_class"]):
            fig.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],
                    mode="markers",
                    marker=dict(size=12, color=MERIT_ORDER_COLORS.get(fuel, "gray"), symbol="square"),
                    name=fuel,
                    legendgroup=fuel,
                    showlegend=True,
                )
            )

    y_max = mo["mc_eur_mwh"].quantile(0.98) * 1.15
    fig.update_layout(
        title=title,
        xaxis_title="Cumulative GW",
        yaxis_title="Marginal Cost (EUR/MWh)",
        height=height,
        hovermode="closest",
        template="plotly_white",
        yaxis=dict(range=[0, y_max]),
        xaxis=dict(range=[0, mo["cum_GW"].max() * 1.02]),
    )
    return fig


def plot_regional_merit_order_plotly(result: ThermalStackResult, height: int = 600):
    """Create East-vs-West merit order charts."""
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    west = build_merit_order(result.delivery_month, result.marginal_costs, result.active_units, region="West")
    east = build_merit_order(result.delivery_month, result.marginal_costs, result.active_units, region="East")
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["West Japan", "East Japan"],
        shared_yaxes=True,
        horizontal_spacing=0.05,
    )

    for column_index, merit_order in enumerate([west, east], start=1):
        shown_legends: set[str] = set()
        for _, row in merit_order.iterrows():
            fuel = row["fuel_class"]
            x0 = row["cum_GW"] - row["capacity_mw"] / 1000
            x1 = row["cum_GW"]
            show_legend = fuel not in shown_legends and column_index == 1
            shown_legends.add(fuel)
            fig.add_trace(
                go.Scatter(
                    x=[x0, x1, x1, x0, x0],
                    y=[0, 0, row["mc_eur_mwh"], row["mc_eur_mwh"], 0],
                    fill="toself",
                    fillcolor=MERIT_ORDER_COLORS.get(fuel, "gray"),
                    opacity=0.7,
                    line=dict(color=MERIT_ORDER_COLORS.get(fuel, "gray"), width=0.5),
                    name=fuel,
                    legendgroup=fuel,
                    showlegend=show_legend,
                    hoverinfo="skip",
                ),
                row=1,
                col=column_index,
            )

    fig.update_xaxes(title_text="Cumulative GW", row=1, col=1)
    fig.update_xaxes(title_text="Cumulative GW", row=1, col=2)
    fig.update_yaxes(title_text="Marginal Cost (EUR/MWh)", row=1, col=1)
    fig.update_layout(
        title=f"Merit Order - West vs East Japan - {result.delivery_month.strftime('%B %Y')}",
        height=height,
        template="plotly_white",
        hovermode="closest",
    )
    return fig


def main() -> None:
    """Run thermal stack pipeline from command line."""
    result = build_thermal_stack()
    print(f"Thermal stack rows: {len(result.merit_order):,}")
    print(f"Delivery month: {result.delivery_month:%Y-%m}")
    if result.export_path is not None:
        print(f"Exported to: {result.export_path}")


if __name__ == "__main__":
    main()

Thermal stack rows: 363
Delivery month: 2026-04
Exported to: C:\Develop\data\Japan\thermalStack.xlsx


## Merit Order - All Japan


In [55]:
fig = plot_merit_order_plotly(
    merit_order = result.merit_order,
    region = "Kansai",
    # period = result.delivery_month,
    title=f"Japan Power Merit Order - {result.delivery_month:%B %Y}",
)
fig.show()


In [46]:
# Single region: Tokyo
fig_tokyo = plot_merit_order_plotly(
    result.merit_order,
    region="Tokyo",
)
fig_tokyo.show()


In [47]:
# Multiple regions: Tokyo + Kansai
fig_combined = plot_merit_order_plotly(
    result.merit_order,
    region=["Tokyo", "Kansai"],
)
fig_combined.show()


In [48]:
# Custom period: Hokkaido + Tokyo
fig_period = plot_merit_order_plotly(
    result.merit_order,
    region=["Hokkaido", "Tokyo"],
    period="2026-03",
)
fig_period.show()


## Merit Order - East vs West


In [49]:
fig = plot_regional_merit_order_plotly(result)
fig.show()


In [50]:
# J-STAGE reference: https://www.jstage.jst.go.jp/article/jsaisigtwo/2026/BI-028/2026_16/_pdf/-char/en
from IPython.display import display
import pandas as pd
import plotly.graph_objects as go

ARTICLE_FUEL_PARAMS = pd.DataFrame(
    [
        {
            "article_fuel": "Oil",
            "stack_fuel": "Oil",
            "article_level": "Thermal fossil",
            "article_parameters": "CV=41.63 MJ/L; C=7600 JPY/kL; alpha=4.8%; eta_i unit-level",
            "article_cost_reference": "Fuel price plus C_f, divided by heat content, efficiency and net auxiliary factor",
            "stack_cost_reference": "JCC EUR/MWh / converted efficiency + Oil VOM",
            "same_efficiency_logic": "Yes: unit-level efficiency; stack converts LHV efficiency to HHV fuel-price basis.",
            "cost_similarity": "Similar variable/marginal-cost intent; article adds explicit auxiliary rate and expense term.",
        },
        {
            "article_fuel": "LNG",
            "stack_fuel": "LNG",
            "article_level": "Thermal fossil",
            "article_parameters": "CV=54.7 MJ/kg; C=2800 JPY/t; alpha=2.3%; eta_i unit-level",
            "article_cost_reference": "JLC/customs reference, with JKM where applicable for some utilities",
            "stack_cost_reference": "JLC EUR/MWh / converted efficiency + LNG VOM",
            "same_efficiency_logic": "Yes: unit-level efficiency; stack converts LHV efficiency to HHV fuel-price basis.",
            "cost_similarity": "Similar variable/marginal-cost intent; article is more explicit on auxiliary losses and expenses.",
        },
        {
            "article_fuel": "Coal",
            "stack_fuel": "Coal",
            "article_level": "Thermal fossil",
            "article_parameters": "CV=26.08 MJ/kg; C=2000 JPY/t; alpha=5.5%; eta_i unit-level",
            "article_cost_reference": "Coal import price, with Australian coal where applicable for some utilities",
            "stack_cost_reference": "Coal CIF EUR/MWh / converted efficiency + Coal VOM",
            "same_efficiency_logic": "Yes: unit-level efficiency; stack converts LHV efficiency to HHV fuel-price basis.",
            "cost_similarity": "Similar variable/marginal-cost intent; article adds explicit auxiliary rate and expense term.",
        },
        {
            "article_fuel": "Hydro",
            "stack_fuel": pd.NA,
            "article_level": "Low-cost supply",
            "article_parameters": "Marginal cost fixed at 0 JPY/kWh in the article",
            "article_cost_reference": "0 JPY/kWh",
            "stack_cost_reference": "Not included in current merit-order stack fuels",
            "same_efficiency_logic": "No direct comparison: hydro has no thermal efficiency in the current stack.",
            "cost_similarity": "Not comparable with the stack variable-cost formula.",
        },
        {
            "article_fuel": "Nuclear",
            "stack_fuel": "Nuclear",
            "article_level": "Low-cost supply",
            "article_parameters": "Marginal cost fixed at 0 JPY/kWh in the article",
            "article_cost_reference": "0 JPY/kWh",
            "stack_cost_reference": "7 USD/MWh VOM / EURUSD; efficiency fixed at 35.1% for stack compatibility",
            "same_efficiency_logic": "No: article does not use nuclear thermal efficiency; stack assigns a fixed placeholder.",
            "cost_similarity": "Not identical: stack gives nuclear a small VOM while the article sets it to zero.",
        },
    ]
)

stack_units = result.active_units.copy()
stack_units["capacity_mw"] = pd.to_numeric(stack_units["capacity_mw"], errors="coerce")
stack_units["plant_efficiency"] = pd.to_numeric(stack_units["plant_efficiency"], errors="coerce")

stack_efficiency_summary = (
    stack_units[stack_units["fuel_class"].isin(["LNG", "Coal", "Oil", "Nuclear"])]
    .groupby("fuel_class", dropna=False)
    .agg(
        stack_units=("plant_name_jp", "count"),
        stack_capacity_gw=("capacity_mw", lambda values: values.sum(min_count=1) / 1000),
        stack_eff_min_pct=("plant_efficiency", "min"),
        stack_eff_median_pct=("plant_efficiency", "median"),
        stack_eff_max_pct=("plant_efficiency", "max"),
    )
    .rename_axis("stack_fuel")
    .reset_index()
)

article_vs_stack = ARTICLE_FUEL_PARAMS.merge(stack_efficiency_summary, on="stack_fuel", how="left")
round_columns = ["stack_capacity_gw", "stack_eff_min_pct", "stack_eff_median_pct", "stack_eff_max_pct"]
article_vs_stack[round_columns] = article_vs_stack[round_columns].round(2)
display(article_vs_stack)

enhanced_merit_order = result.merit_order.copy()
enhanced_merit_order["base_fuel_class"] = enhanced_merit_order["fuel_class"]
enhanced_merit_order["capacity_mw"] = pd.to_numeric(enhanced_merit_order["capacity_mw"], errors="coerce")
enhanced_merit_order["efficiency_pct"] = pd.to_numeric(enhanced_merit_order["efficiency_pct"], errors="coerce")

enhanced_merit_order["fuel_efficiency_band"] = "base"
for fuel in ["Coal", "LNG", "Oil"]:
    fuel_mask = enhanced_merit_order["base_fuel_class"].eq(fuel)
    valid_efficiency = fuel_mask & enhanced_merit_order["efficiency_pct"].notna()
    if not valid_efficiency.any():
        enhanced_merit_order.loc[fuel_mask, "fuel_efficiency_band"] = "unknown efficiency"
        continue

    ranked_efficiency = enhanced_merit_order.loc[valid_efficiency, "efficiency_pct"].rank(method="first", pct=True)
    efficiency_bands = pd.cut(
        ranked_efficiency,
        bins=[0.0, 1 / 3, 2 / 3, 1.0],
        labels=["low efficiency", "mid efficiency", "high efficiency"],
        include_lowest=True,
    ).astype("string")
    enhanced_merit_order.loc[valid_efficiency, "fuel_efficiency_band"] = efficiency_bands.to_numpy()
    enhanced_merit_order.loc[fuel_mask & ~valid_efficiency, "fuel_efficiency_band"] = "unknown efficiency"
split_fuel_mask = enhanced_merit_order["base_fuel_class"].isin(["Coal", "LNG", "Oil"])
enhanced_merit_order.loc[split_fuel_mask, "fuel_class"] = (
    enhanced_merit_order.loc[split_fuel_mask, "base_fuel_class"]
    + " - "
    + enhanced_merit_order.loc[split_fuel_mask, "fuel_efficiency_band"].astype(str)
)

enhanced_level_summary = (
    enhanced_merit_order.groupby("fuel_class", dropna=False)
    .agg(
        units=("plant_name_jp", "count"),
        capacity_gw=("capacity_mw", lambda values: values.sum(min_count=1) / 1000),
        efficiency_min_pct=("efficiency_pct", "min"),
        efficiency_median_pct=("efficiency_pct", "median"),
        efficiency_max_pct=("efficiency_pct", "max"),
        marginal_cost_median_eur_mwh=("mc_eur_mwh", "median"),
    )
    .sort_values("marginal_cost_median_eur_mwh")
    .round(2)
)
display(enhanced_level_summary)

# plot_merit_order_plotly expects the base fuel labels, so build with base labels and restyle by efficiency band.
enhanced_plot_input = enhanced_merit_order.copy()
enhanced_plot_input["fuel_class"] = enhanced_plot_input["base_fuel_class"]
fig = plot_merit_order_plotly(
    enhanced_plot_input,
    title=f"Japan Power Merit Order - efficiency bands - {result.delivery_month:%B %Y}",
)

enhanced_colors = {
    "Nuclear": "#9467bd",
    "Coal - high efficiency": "#111111",
    "Coal - mid efficiency": "#555555",
    "Coal - low efficiency": "#9a9a9a",
    "LNG - high efficiency": "#c85500",
    "LNG - mid efficiency": "#ff8c00",
    "LNG - low efficiency": "#ffd08a",
    "Oil - high efficiency": "#5a2c0c",
    "Oil - mid efficiency": "#8b4513",
    "Oil - low efficiency": "#c08a5a",
}
plot_row_order = pd.concat(
    [
        enhanced_merit_order[enhanced_merit_order["base_fuel_class"] == fuel]
        for fuel in ["Nuclear", "Coal", "LNG", "Oil"]
    ],
    ignore_index=True,
)
for trace, (_, row) in zip(fig.data[: len(plot_row_order)], plot_row_order.iterrows()):
    label = row["fuel_class"]
    color = enhanced_colors.get(label, "gray")
    trace.update(fillcolor=color, line=dict(color=color, width=0.5), legendgroup=label, name=label)

for trace in fig.data[len(plot_row_order):]:
    trace.update(showlegend=False)

legend_order = [
    "Nuclear",
    "Coal - high efficiency", "Coal - mid efficiency", "Coal - low efficiency",
    "LNG - high efficiency", "LNG - mid efficiency", "LNG - low efficiency",
    "Oil - high efficiency", "Oil - mid efficiency", "Oil - low efficiency",
]
present_labels = set(enhanced_merit_order["fuel_class"])
for label in [label for label in legend_order if label in present_labels]:
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(size=12, color=enhanced_colors.get(label, "gray"), symbol="square"),
            name=label,
            legendgroup=label,
            showlegend=True,
        )
    )
fig.update_layout(legend_title_text="Fuel / efficiency band")
fig.show()


,article_fuel,stack_fuel,article_level,article_parameters,article_cost_reference,stack_cost_reference,same_efficiency_logic,cost_similarity,stack_units,stack_capacity_gw,stack_eff_min_pct,stack_eff_median_pct,stack_eff_max_pct
0,Oil,Oil,Thermal fossil,CV=41.63 MJ/L; C=7600 JPY/kL; alpha=4.8%; eta_...,"Fuel price plus C_f, divided by heat content, ...",JCC EUR/MWh / converted efficiency + Oil VOM,Yes: unit-level efficiency; stack converts LHV...,Similar variable/marginal-cost intent; article...,45.0,9.69,38.74,38.85,39.59
1,LNG,LNG,Thermal fossil,CV=54.7 MJ/kg; C=2800 JPY/t; alpha=2.3%; eta_i...,"JLC/customs reference, with JKM where applicab...",JLC EUR/MWh / converted efficiency + LNG VOM,Yes: unit-level efficiency; stack converts LHV...,Similar variable/marginal-cost intent; article...,288.0,81.86,34.49,55.52,60.48
2,Coal,Coal,Thermal fossil,CV=26.08 MJ/kg; C=2000 JPY/t; alpha=5.5%; eta_...,"Coal import price, with Australian coal where ...",Coal CIF EUR/MWh / converted efficiency + Coal...,Yes: unit-level efficiency; stack converts LHV...,Similar variable/marginal-cost intent; article...,155.0,52.73,39.24,39.24,41.56
3,Hydro,NaN,Low-cost supply,Marginal cost fixed at 0 JPY/kWh in the article,0 JPY/kWh,Not included in current merit-order stack fuels,No direct comparison: hydro has no thermal eff...,Not comparable with the stack variable-cost fo...,NaN,NaN,NaN,NaN,NaN
4,Nuclear,Nuclear,Low-cost supply,Marginal cost fixed at 0 JPY/kWh in the article,0 JPY/kWh,7 USD/MWh VOM / EURUSD; efficiency fixed at 35...,No: article does not use nuclear thermal effic...,Not identical: stack gives nuclear a small VOM...,18.0,17.98,35.10,35.10,35.10


,units,capacity_gw,efficiency_min_pct,efficiency_median_pct,efficiency_max_pct,marginal_cost_median_eur_mwh
fuel_class,,,,,,
Nuclear,18,17.98,35.10,35.10,35.10,5.98
LNG - high efficiency,100,47.41,55.52,57.01,60.48,61.38
LNG - mid efficiency,100,12.44,55.52,55.52,55.52,62.96
LNG - low efficiency,100,25.74,34.49,41.39,55.52,83.57
Oil - high efficiency,15,2.58,38.85,38.85,39.59,95.55
Oil - mid efficiency,15,4.92,38.85,38.85,38.85,95.55
Oil - low efficiency,15,2.19,38.74,38.74,38.85,95.80
